In [1]:
import QuantLib as ql

In [2]:
today = ql.Date(8, ql.October, 2014)
ql.Settings.instance().evaluationDate = today

<h4>A somewhat exotic option</h4>

As an example, we’ll use a knock-in barrier option:

In [3]:
option = ql.BarrierOption(ql.Barrier.UpIn,
                          120.0, #barrier
                          0.0, #rebate
                          ql.PlainVanillaPayoff(ql.Option.Call, 100.0),
                          ql.EuropeanExercise(ql.Date(8, ql.January, 2015))
    
)

For the purpose of this example, the market data are the underlying value, the risk-free rate and the volatility. We wrap them in quotes, so that the instrument will be notified of any changes…

In [4]:
u = ql.SimpleQuote(100.0)
r = ql.SimpleQuote(0.01)
sigma = ql.SimpleQuote(0.20)

from the quotes we build the flat curves and the process that the engine requires.

In [5]:
riskFreeCurve = ql.FlatForward(0, ql.TARGET(),
                               ql.QuoteHandle(r), ql.Actual360())
volatility = ql.BlackConstantVol(0, ql.TARGET(),
                                ql.QuoteHandle(sigma), ql.Actual360())

In [6]:
process = ql.BlackScholesProcess(ql.QuoteHandle(u),
                                 ql.YieldTermStructureHandle(riskFreeCurve),
                                 ql.BlackVolTermStructureHandle(volatility))

In [7]:
option.setPricingEngine(ql.AnalyticBarrierEngine(process))

In [8]:
print(option.NPV())

1.3657980739109867


…but we’re not so lucky when it comes to Greeks:

In [9]:
#print(option.delta()) #RuntimeError: delta not provides

<h4>Numerical calculation</h4>

We just have
to set the relevant quote to the new value and ask the option for its price again. Thus, we choose a
small increment and start.

In [10]:
u0= u.value(); h = 0.01

In [11]:
P0 = option.NPV(); print(P0)

1.3657980739109867


…then we increase the underlying value and get the new option value…

In [12]:
u.setValue(u0 + h)
P_plus = option.NPV(); print(P_plus)

1.3688112201958078


…then we do the same after decreasing the underlying value.

In [14]:
u.setValue(u0 - h)
P_minus = option.NPV(); print(P_minus)

1.3627900998610203


Finally, we set the underlying value back to its current value.

u.setValue(u0)

Applying the formulas above give us the desired Greeks:

In [16]:
Delta = (P_plus - P_minus)/(2*h)
Gamma= (P_plus - 2*P0 + P_minus)/(h*h)
print(Delta)
print(Gamma)

0.3010560167393761
0.05172234854633473
